# Model Training

## Objective

The objective of this notebook is to train multiple machine learning models on the engineered fraud detection features and compare their performance.

The best-performing model will be selected for deployment in the Fake Internship & Job Scam Detection System.

Section 1: Importing Libraries

In [1]:
import joblib
import pandas as pd

from scipy.sparse import load_npz

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score

In [10]:
import os

print(os.path.getmtime("../data/processed/X_train.npz"))
print(os.path.getmtime("../data/processed/X_test.npz"))

1780922850.5615652
1780922850.6593895


Section 2 : Loading Prepared data

In [11]:
X_train = load_npz("../data/processed/X_train.npz")
X_test = load_npz("../data/processed/X_test.npz")

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

print(X_train.shape)
print(X_test.shape)

(5876, 5006)
(1469, 5006)


Section 3 : Training Logistic Regression

In [12]:
lr_model = LogisticRegression(
    random_state=42,
    max_iter=6000,
    class_weight="balanced"
)

lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)

lr_accuracy = accuracy_score(y_test, lr_pred)

print(f"Logistic Regression Accuracy: {lr_accuracy:.4f}")

Logistic Regression Accuracy: 0.9973


Section 4 : Random Forest Classifier

In [13]:
from sklearn.ensemble import RandomForestClassifier

# Initialize Model
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

# Train Model
rf_model.fit(X_train, y_train)

# Predictions
rf_pred = rf_model.predict(X_test)

# Accuracy
rf_accuracy = accuracy_score(y_test, rf_pred)

print(f"Random Forest Accuracy : {rf_accuracy:.4f}")

Random Forest Accuracy : 0.9966


Section 5 : XGBOOST Classifier

In [14]:
from xgboost import XGBClassifier

# Initialize Model
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

# Train Model
xgb_model.fit(X_train, y_train)

# Predictions
xgb_pred = xgb_model.predict(X_test)

# Accuracy
xgb_accuracy = accuracy_score(y_test, xgb_pred)

print(f"XGBoost Accuracy : {xgb_accuracy:.4f}")

XGBoost Accuracy : 0.9980


In [62]:
import joblib
import os

# Create models directory if it doesn't exist
os.makedirs("../models", exist_ok=True)

# Save Logistic Regression Model
joblib.dump(lr_model, "../models/lr_model.pkl")
print("Logistic Regression model saved successfully.")

# Save Random Forest Model
joblib.dump(rf_model, "../models/rf_model.pkl")
print("Random Forest model saved successfully.")

# Save XGBoost Model
joblib.dump(xgb_model, "../models/xgb_model.pkl")
print("XGBoost model saved successfully.")


Logistic Regression model saved successfully.
Random Forest model saved successfully.
XGBoost model saved successfully.


Data Leakage Chekcing due to high accuracy getting

In [15]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, lr_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, lr_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1305
           1       1.00      0.98      0.99       164

    accuracy                           1.00      1469
   macro avg       1.00      0.99      0.99      1469
weighted avg       1.00      1.00      1.00      1469


Confusion Matrix:

[[1305    0]
 [   4  160]]


In [16]:
from sklearn.metrics import roc_auc_score

probs = lr_model.predict_proba(X_test)[:,1]

print("ROC-AUC:", roc_auc_score(y_test, probs))

ROC-AUC: 0.9897159144005234


Section: Custom Job Prediction Test

In [63]:
import joblib
from scipy.sparse import load_npz
import pandas as pd

tfidf     = joblib.load("../models/tfidf_vectorizer.pkl")
lr_model  = joblib.load("../models/lr_model.pkl")
rf_model  = joblib.load("../models/rf_model.pkl")
xgb_model = joblib.load("../models/xgb_model.pkl")

df_ref = pd.read_csv("../data/processed/feature_engineered_dataset.csv")
DOMAIN_FREQ_MEDIAN = float(df_ref["domain_frequency"].median())

print(" All artifacts loaded")
print(f"   TF-IDF vocab size      : {len(tfidf.vocabulary_)}")
print(f"   domain_frequency median: {DOMAIN_FREQ_MEDIAN}")

 All artifacts loaded
   TF-IDF vocab size      : 5000
   domain_frequency median: 6553.0


In [70]:
import re, html, unicodedata
from scipy.sparse import csr_matrix, hstack

fraud_keywords = [
    "registration fee", "training fee", "whatsapp", "telegram",
    "earn daily", "instant joining", "limited seats", "quick money",
    "no interview", "work from home", "guaranteed placement"
]

salary_keywords = [
    "earn", "daily income", "easy money",
    "part time income", "guaranteed earnings"
]


def clean_combined_text(text: str) -> str:
    """Exact same pipeline as notebook 03 clean_combined_text()"""
    if not text:
        return ""
    text = html.unescape(str(text))
    text = text.lower()
    text = unicodedata.normalize("NFKD", text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def predict_job_v3(
    job_description : str,
    job_title       : str = "",
    company_name    : str = "",
    field           : str = "",
    experience_level: str = ""
) -> None:

    # Step 1: combined_text — same as notebook 03
    raw_combined = (
        job_title + " " + company_name + " " +
        field     + " " + experience_level + " " + job_description
    )

    # Step 2: Clean — same pipeline as training
    text = clean_combined_text(raw_combined)

    # Step 3: TF-IDF (fitted on combined_text in notebook 05)
    X_text = tfidf.transform([text])               # (1, 5000)

    # Step 4: Numerical features — same order as notebook 05
    words        = text.split()
    text_length  = len(text)
    word_count   = len(words)
    avg_word_len = sum(len(w) for w in words) / word_count if word_count > 0 else 0.0
    kw_risk      = sum(kw in text for kw in fraud_keywords)
    sal_risk     = int(any(kw in text for kw in salary_keywords))

    X_num = csr_matrix([[
        text_length, word_count, avg_word_len,
        kw_risk, sal_risk, DOMAIN_FREQ_MEDIAN
    ]])                                            # (1, 6)

    # Step 5: hstack → (1, 5006)
    X_final = hstack([X_text, X_num])

    # Step 6: Predict
    print(f"\n{'='*48}")
    print(f"  Processed text (80 chars): {text[:80]}...")
    print(f"  keyword_risk : {kw_risk}  |  salary_risk : {sal_risk}")
    print(f"{'='*48}")
    for name, model in {
        "Logistic Regression" : lr_model,
        "Random Forest"       : rf_model,
        "XGBoost"             : xgb_model,
    }.items():
        pred  = model.predict(X_final)[0]
        proba = model.predict_proba(X_final)[0]
        conf  = round(max(proba) * 100, 2)
        label = "🟢 REAL JOB" if pred == 0 else "🔴 FAKE JOB"
        print(f"  {name:22}: {label}  ({conf}%)")
    print(f"{'='*48}\n")

In [71]:
predict_job_v3(
    job_title        = "Data Science Intern",
    company_name     = "Infosys Ltd.",
    field            = "Information Technology",
    experience_level = "Entry Level",
    job_description  = """
    Infosys is looking for Data Science interns to assist in predictive
    analytics and AI research. Knowledge of Python, SQL, Pandas required.
    Selection: Online Assessment → Technical Interview → HR Interview
    Apply on official Infosys Careers Portal. Stipend 30000 per month.
    """
)


  Processed text (80 chars): data science intern infosys ltd information technology entry level infosys is lo...
  keyword_risk : 0  |  salary_risk : 0
  Logistic Regression   : 🟢 REAL JOB  (99.09%)
  Random Forest         : 🟢 REAL JOB  (81.0%)
  XGBoost               : 🟢 REAL JOB  (99.9800033569336%)



In [72]:
predict_job_v3(
    job_title       = "Work From Home Opportunity",
    company_name    = "QuickEarn Solutions",
    job_description = """
    Earn 5000 daily from home! No interview required. Instant joining.
    Pay registration fee of 500 via WhatsApp to get started.
    Guaranteed placement. Limited seats. Contact on Telegram.
    Quick money, easy money, guaranteed earnings every week.
    """
)


  Processed text (80 chars): work from home opportunity quickearn solutions earn daily from home no interview...
  keyword_risk : 10  |  salary_risk : 1
  Logistic Regression   : 🟢 REAL JOB  (99.93%)
  Random Forest         : 🟢 REAL JOB  (70.0%)
  XGBoost               : 🟢 REAL JOB  (99.97000122070312%)

